# FFAI Multi-Turn Named Prompt Sequence

This notebook demonstrates FFAI's core capabilities:

1. **Named prompt registration** -- assign logical names to prompts via `prompt_name`
2. **Declarative interpolation** -- reference earlier responses with `{{name.response}}`
3. **Dependency chaining** -- build multi-step reasoning pipelines
4. **History injection** -- supply named prior turns as conversation context via `history=`
5. **DataFrame inspection** -- export interaction history as Polars DataFrames

The model used is Mistral Small via LiteLLM.

In [ ]:
# --- Setup: discover project root, import FFAI, initialize client ---
import sys
from pathlib import Path

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

import os  # noqa: E402

from ffai.Clients import FFLiteLLMClient  # noqa: E402
from ffai.FFAI import FFAI  # noqa: E402

api_key = os.getenv('MISTRAL_API_KEY')
if not api_key:
    raise RuntimeError('Set MISTRAL_API_KEY in your environment')

client = FFLiteLLMClient(
    model_string='mistral/mistral-small-latest',
    api_key=api_key,
    temperature=0.7,
    max_tokens=300,
)

ffai = FFAI(client)
print(f'FFAI initialized with model: {client.model}')

<div class="page-break"></div>

---

## Turn 1: Seed the Topic

Register the first prompt with the name `"languages"`. The response is stored
in FFAI's internal history and can be referenced by name in subsequent turns.

In [ ]:
# Turn 1: ask the model to list three programming languages
# The prompt_name "languages" lets us reference this response later
r1 = ffai.generate_response(
    prompt='Name exactly three programming languages and one sentence about each.',
    prompt_name='languages',
)

print(f'Response:  {r1.response}')
print(f'Model:     {r1.model}')
print(f'Tokens:    {r1.usage}')
print(f'Cost:      ${r1.cost_usd:.6f}')
print(f'Duration:  {r1.duration_ms:.0f}ms')

<div class="page-break"></div>

---

## Turn 2: Interpolation -- Reference Turn 1

FFAI resolves `{{languages.response}}` by looking up the latest response stored
under the `"languages"` prompt name and substituting it into the prompt template.
This is the core **declarative context assembly** mechanism -- no manual string
concatenation needed.

In [ ]:
# Turn 2: {{languages.response}} is replaced with the actual response from Turn 1
# before the prompt is sent to the model
r2 = ffai.generate_response(
    prompt='Which of {{languages.response}} is best suited for data science, and why?',
    prompt_name='recommendation',
)

print(f'Response:  {r2.response}')
print(f'Model:     {r2.model}')
print(f'Tokens:    {r2.usage}')
print(f'Cost:      ${r2.cost_usd:.6f}')
print(f'Duration:  {r2.duration_ms:.0f}ms')

<div class="page-break"></div>

---

## Turn 3: Multi-Dependency Interpolation

Multiple `{{name.response}}` patterns can appear in a single prompt. FFAI resolves
each one independently. Here we chain both `{{languages.response}}` and
`{{recommendation.response}}` into a single prompt to produce a learning plan.

In [ ]:
# Turn 3: two interpolation patterns resolved in one prompt
r3 = ffai.generate_response(
    prompt=(
        'Given {{languages.response}}, and the recommendation that '
        '{{recommendation.response}}, write a one-paragraph learning plan '
        'for a beginner starting with that language.'
    ),
    prompt_name='learning_plan',
)

print(f'Response:  {r3.response}')
print(f'Model:     {r3.model}')
print(f'Tokens:    {r3.usage}')
print(f'Cost:      ${r3.cost_usd:.6f}')
print(f'Duration:  {r3.duration_ms:.0f}ms')

<div class="page-break"></div>

---

## Turn 4: History-Based Context Injection

The `history=` parameter provides a different context mechanism from interpolation.
Instead of substituting text inline, it wraps the named prior turns in
`<conversation_history>` XML tags and prepends them to the prompt. This gives the
model full conversational context for synthesis tasks like summarization.

In [ ]:
# Turn 4: inject all three prior named turns as conversation context
# The history parameter wraps prior interactions in <conversation_history> tags
r4 = ffai.generate_response(
    prompt='Summarize everything discussed so far in two sentences.',
    prompt_name='summary',
    history=['languages', 'recommendation', 'learning_plan'],
)

print(f'Response:  {r4.response}')
print(f'Model:     {r4.model}')
print(f'Tokens:    {r4.usage}')
print(f'Cost:      ${r4.cost_usd:.6f}')
print(f'Duration:  {r4.duration_ms:.0f}ms')

<div class="page-break"></div>

---

## Inspecting Conversation History

FFAI maintains a raw conversation history that records every interaction.
The `history_to_dataframe()` method converts it to a Polars DataFrame with
columns for prompt_name, model, response, and datetime.

In [ ]:
# Export raw conversation history as a Polars DataFrame
conv_df = ffai.history_to_dataframe()

cols = [c for c in ['prompt_name', 'model', 'response', 'datetime'] if c in conv_df.columns]
print(f'Conversation history: {conv_df.shape[0]} rows x {conv_df.shape[1]} columns')
print()
print(conv_df.select(cols))

<div class="page-break"></div>

---

## Inspecting Ordered Prompt History

The ordered history provides a sequenced view of all interactions, including
the original prompt text (before interpolation), the resolved response, and
a sequence number for ordering.

In [ ]:
# Export ordered prompt history as a Polars DataFrame
ordered_df = ffai.ordered_history_to_dataframe()

cols = [
    c
    for c in ['sequence_number', 'prompt_name', 'prompt', 'response', 'model', 'datetime']
    if c in ordered_df.columns
]
print(f'Ordered history: {ordered_df.shape[0]} rows x {ordered_df.shape[1]} columns')
print()
print(ordered_df.select(cols))

<div class="page-break"></div>

---

## Prompt Name Usage Statistics

FFAI tracks how many times each prompt name was used, which is useful for
understanding the shape of a multi-step pipeline.

In [ ]:
# Show usage counts per prompt name
stats_df = ffai.get_prompt_name_stats_df()
print(stats_df)

<div class="page-break"></div>

---

## Cost and Token Summary

Aggregate token usage and cost across all four turns.

In [ ]:
# Aggregate cost and token usage from all four turns
total_input = r1.usage.input_tokens + r2.usage.input_tokens + r3.usage.input_tokens + r4.usage.input_tokens
total_output = r1.usage.output_tokens + r2.usage.output_tokens + r3.usage.output_tokens + r4.usage.output_tokens
total_cost = r1.cost_usd + r2.cost_usd + r3.cost_usd + r4.cost_usd
total_duration = r1.duration_ms + r2.duration_ms + r3.duration_ms + r4.duration_ms

print(f'Total input tokens:   {total_input}')
print(f'Total output tokens:  {total_output}')
print(f'Total cost:           ${total_cost:.6f}')
print(f'Total wall time:      {total_duration:.0f}ms ({total_duration / 1000:.1f}s)')